# 00 - Prepare after Colab reset (Run All once, then35)

**Normal workflow after every reset: this notebook ->35.** Do NOT rerun01/02/04/23/29/33/34.
Select T4 if available;35 also supports CPU but can be slow. This notebook is self-contained:
it embeds the current35 READ-ONLY preflight, not a path to another uploaded notebook.

What Run All does:
1. Mount Drive. Read existing frozen configurations; restore required package versions only when
   missing/different, with explicit pip errors. Never rewrite a frozen config to accept a new version.
2. Restore ORIGINAL TRAIN and development-validation image BYTES into runtime from the pinned archive.
   No image decoding/quality scoring, split generation or audit recreation. Held-out synthetic test,
   FVC2004 and quarantined images are NOT extracted. Matching/35 will still enforce their own protocols.
3. Reuse verified Drive archive; download only if missing. Reuse correct runtime files; explicitly
   repair missing/changed runtime copies atomically. No frozen Drive image/checkpoint is replaced.
4. Restore only missing PUBLIC DMD source/weights from the exact pinned links/checksums. Reuse existing
   checkpoints and33/34 caches; missing research artifacts stop with their path, never retrain.
5. Run current35 prerequisite checks (including original23 warmstart), then a small CPU/CUDA smoke check.
   Do NOT scan every33 per-image cache over Drive here:35 verifies those bundles when it uses them.
   Setup readiness does not attest that all per-image caches are present;35 stops on missing/changed caches.
   Each prerequisite stage reports elapsed time. No optimizer, training, recognition or new gate.
6. Optional NBIS install/version check: switch `PREPARE_NBIS=True` below for later NBIS matching work.
   Default False because35 does NOT need it. This reuses04 build commands but not its biometric smoke test.

If Colab changed preloaded packages, a ONE-TIME kernel restart may be unavoidable after pip restores
the recorded versions. The notebook will STOP explicitly; use Runtime > Restart session, then Run All
THIS notebook again. No manual package cells or old training notebooks. If the official frozen Torch
wheel does not support the selected Python, installation stops; no fake compatibility or version relaxation.

Preparation changes only ephemeral runtime files and missing verified public download caches on Drive.
It NEVER changes split/audit reports, reference galleries, experiment configs, trained weights,33/34
scores or gate outcomes. In particular29/34 stay failed and30/31 remain blocked.
This only prepares the current35 workflow, not authorization for training or held-out test evaluation.


In [ ]:

import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import ast
import csv
import gc
import hashlib
import importlib.metadata
import io
import json
import platform
import re
import shutil
import subprocess
import sys
import time
import urllib.request
import zipfile
from pathlib import Path, PurePosixPath
from google.colab import drive

PREPARE_NBIS = False
drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/FingerprintResearch')
LOCAL_ROOT = Path('/content/fingerprint_research')
DATA_ROOT = LOCAL_ROOT/'data'
PREPARE_DIR = LOCAL_ROOT/'prepare_after_reset'
for directory in (LOCAL_ROOT,DATA_ROOT,PREPARE_DIR):
    directory.mkdir(parents=True,exist_ok=True)
READY_STAMP = PREPARE_DIR/'ready_for_35.json'
if READY_STAMP.exists():
    READY_STAMP.unlink()
print('Prepare only: no training/reselection/split/gate updates.')


In [ ]:
PREPARE_SOURCE = 'def file_digest(path,algorithm=\'sha256\'):\n    digest=hashlib.new(algorithm)\n    with path.open(\'rb\') as stream:\n        for chunk in iter(lambda:stream.read(8*1024*1024),b\'\'):\n            digest.update(chunk)\n    return digest.hexdigest()\n\ndef read_json(path):\n    if not path.is_file():\n        raise FileNotFoundError(f\'Missing persistent artifact: {path}. This is not a runtime reset loss; no training will be rerun automatically.\')\n    return json.loads(path.read_text(encoding=\'utf-8\'))\n\ndef safe_relative(relative):\n    if not isinstance(relative,str) or not relative or \'\\\\\' in relative or \':\' in relative:\n        raise RuntimeError(f\'Unsafe frozen relative path: {relative!r}\')\n    path=PurePosixPath(relative)\n    if not path.parts or path.is_absolute() or \'..\' in path.parts or path.as_posix()!=relative:\n        raise RuntimeError(f\'Unsafe frozen relative path: {relative!r}\')\n    return path\n\ndef safe_destination(root,relative):\n    destination=root/Path(*safe_relative(relative).parts)\n    if root.resolve() not in destination.resolve().parents:\n        raise RuntimeError(f\'Runtime path escapes its declared root: {relative}\')\n    return destination\n\ndef verify_manifest_files(directory,manifest):\n    for relative,sha in manifest.items():\n        path=directory/Path(*safe_relative(relative).parts)\n        if not path.is_file() or file_digest(path)!=sha:\n            raise RuntimeError(f\'Persistent artifact missing/changed: {path}. Do not regenerate or relax its checksum.\')\n\ndef installed_version(name):\n    try:\n        return importlib.metadata.version(name)\n    except importlib.metadata.PackageNotFoundError:\n        return None\n\ndef restore_packages(requirements,log_directory,metadata_version=installed_version,runner=subprocess.run,loaded=None):\n    loaded=sys.modules if loaded is None else loaded\n    module_names={\'numpy\':\'numpy\',\'Pillow\':\'PIL\',\'scipy\':\'scipy\',\'torch\':\'torch\',\'pandas\':\'pandas\',\'matplotlib\':\'matplotlib\'}\n    changes=[]\n    for name,version in requirements.items():\n        if name not in module_names or (version is not None and not re.fullmatch(r\'[0-9][0-9A-Za-z.+_-]*\',version)):\n            raise RuntimeError(\'Invalid frozen dependency requirement.\')\n        found=metadata_version(name)\n        if found is None or (version is not None and found!=version):\n            changes.append((name,version,found))\n    if not changes:\n        print(\'Recorded core package versions already available; no pip installs.\')\n        return dict(changed=[],restart_required=False)\n    constraints=log_directory/\'frozen_requirements.constraints.txt\'\n    constraints.write_text(\'\'.join(f\'{name}=={version}\\n\' for name,version in requirements.items()\n        if version is not None),encoding=\'utf-8\')\n    restart=False\n    for name,version,found in changes:\n        print(f\'Required dependency unavailable: {name}, installed={found}, required={version or "installed"}; restoring.\')\n        command=[sys.executable,\'-m\',\'pip\',\'install\',\'--disable-pip-version-check\',\'--no-input\',\n            \'--constraint\',str(constraints)]\n        command.append(f\'{name}=={version}\' if version is not None else name)\n        if name==\'torch\':\n            match=re.search(r\'\\+cu([0-9]+)$\',version or \'\')\n            if match:\n                command.extend([\'--index-url\',f\'https://download.pytorch.org/whl/cu{match.group(1)}\'])\n            elif version and version.endswith(\'+cpu\'):\n                command.extend([\'--index-url\',\'https://download.pytorch.org/whl/cpu\'])\n        log=log_directory/f\'pip_{name}.log\'\n        with log.open(\'w\',encoding=\'utf-8\') as output:\n            result=runner(command,stdout=output,stderr=subprocess.STDOUT,text=True)\n        if result.returncode:\n            print(\'\\n\'.join(log.read_text(encoding=\'utf-8\',errors=\'replace\').splitlines()[-35:]))\n            raise RuntimeError(f\'Cannot restore frozen {name}. See {log}; check Python/wheel compatibility. No version fallback or training was run.\')\n        importlib.invalidate_caches()\n        restart=restart or module_names[name] in loaded\n    for name,version in requirements.items():\n        actual=metadata_version(name)\n        if actual is None or (version is not None and actual!=version):\n            raise RuntimeError(f\'Dependency restore did not produce required {name}=={version}; got {actual}.\')\n    return dict(changed=[name for name,_,_ in changes],restart_required=restart)\n\ndef frozen_environment(project,source33_id):\n    split_dir=project/\'protocols\'/\'fvc_split_v1\'\n    split=read_json(split_dir/\'protocol.json\')\n    if split[\'version\']!=\'fvc_split_v1\':\n        raise RuntimeError(\'Expected existing frozen split, not a new split.\')\n    verify_manifest_files(split_dir,split[\'file_sha256\'])\n    m0=read_json(project/\'experiments\'/\'m0_unet_v1_seed20261110\'/\'configuration.json\')\n    m1=read_json(project/\'experiments\'/\'m1_shared_curriculum_v1_seed20261110\'/\'dd1427cfad6621c577bf\'/\'configuration.json\')\n    source_dir=project/\'diagnostics\'/\'pretrained_dmd_feasibility_v1\'/source33_id\n    source=read_json(source_dir/\'configuration.json\')\n    completion=read_json(source_dir/\'completion.json\')\n    sha=file_digest(source_dir/\'configuration.json\')\n    if (source_dir.name!=sha[:20] or completion[\'configuration_sha256\']!=sha\n            or completion[\'status\']!=\'pretrained_dmd_feasibility_complete\'):\n        raise RuntimeError(\'Completed33 identity/integrity differs.\')\n    verify_manifest_files(source_dir,completion[\'file_sha256\'])\n    for key in (\'torch_version\',\'torch_cuda\',\'numpy_version\',\'pillow_version\'):\n        if m0[key]!=m1[key] or m0[key]!=source[key]:\n            raise RuntimeError(f\'Frozen experiments disagree about {key}; do not auto-pick a version.\')\n    requirements={\'torch\':source[\'torch_version\'],\'numpy\':source[\'numpy_version\'],\n        \'Pillow\':source[\'pillow_version\'],\'scipy\':source[\'scipy_version\'],\'pandas\':None,\'matplotlib\':None}\n    return requirements,source,split\n\ndef locked_development_records(split_dir):\n    protocol=read_json(split_dir/\'protocol.json\')\n    verify_manifest_files(split_dir,protocol[\'file_sha256\'])\n    with (split_dir/\'manifest_splits.csv\').open(encoding=\'utf-8\',newline=\'\') as stream:\n        all_records=list(csv.DictReader(stream))\n    if len(all_records)!=7920 or len({row[\'relative_path\'] for row in all_records})!=len(all_records):\n        raise RuntimeError(\'Unexpected frozen FVC manifest count/duplicate paths.\')\n    selected=[]\n    for row in all_records:\n        safe_relative(row[\'relative_path\'])\n        if not re.fullmatch(r\'[0-9a-f]{64}\',row[\'sha256\']):\n            raise RuntimeError(\'Invalid frozen image SHA256.\')\n        if row[\'split\'] in (\'train\',\'validation\'):\n            if row[\'dataset\'] not in (\'FVC2000\',\'FVC2002\'):\n                raise RuntimeError(\'TRAIN/development restoration must not include external/test data.\')\n            if not re.fullmatch(r\'FVC(?:2000|2002)/DB[123]_[AB]/[0-9]+_[0-9]+\\.(?:tif|bmp)\',row[\'relative_path\'],re.IGNORECASE):\n                raise RuntimeError(\'Unexpected frozen development image path.\')\n            selected.append(row)\n    if not selected or {row[\'split\'] for row in selected}!={\'train\',\'validation\'}:\n        raise RuntimeError(\'Missing frozen TRAIN/development-validation partitions.\')\n    return selected\n\ndef copy_checked(source,destination,expected_bytes,expected_md5):\n    destination.parent.mkdir(parents=True,exist_ok=True)\n    pending=destination.with_suffix(destination.suffix+\'.prepare.part\')\n    with source.open(\'rb\') as src,pending.open(\'wb\') as dst:\n        shutil.copyfileobj(src,dst,8*1024*1024)\n    if pending.stat().st_size!=expected_bytes or file_digest(pending,\'md5\')!=expected_md5:\n        raise RuntimeError(f\'Archive copy failed integrity: {pending}\')\n    pending.replace(destination)\n\ndef prepare_archive(project,local,url,expected_bytes,expected_md5,resumer):\n    drive_archive=project/\'archives\'/\'fvc2000_2002_2004_official.zip\'\n    runtime_archive=local/\'prepare_after_reset\'/\'fvc_official.zip\'\n    runtime_archive.parent.mkdir(parents=True,exist_ok=True)\n    if runtime_archive.exists():\n        if runtime_archive.stat().st_size==expected_bytes and file_digest(runtime_archive,\'md5\')==expected_md5:\n            print(\'Reusing verified local archive.\')\n            return runtime_archive\n        print(\'Repairing changed ephemeral archive from the authoritative cache.\')\n    if drive_archive.exists():\n        if drive_archive.stat().st_size!=expected_bytes:\n            raise RuntimeError(f\'Persistent archive size differs: {drive_archive}. No replacement was made.\')\n        print(\'Copying verified persistent archive to local runtime; no internet download.\',flush=True)\n        copy_checked(drive_archive,runtime_archive,expected_bytes,expected_md5)\n        return runtime_archive\n    partial=runtime_archive.with_suffix(\'.download.part\')\n    print(\'Persistent archive absent: downloading the original pinned public archive.\',flush=True)\n    resumer(url,partial,expected_bytes)\n    if partial.stat().st_size!=expected_bytes or file_digest(partial,\'md5\')!=expected_md5:\n        raise RuntimeError(\'Public FVC archive integrity failure; no images extracted.\')\n    partial.replace(runtime_archive)\n    copy_checked(runtime_archive,drive_archive,expected_bytes,expected_md5)\n    return runtime_archive\n\ndef restore_development_images(records,data_root,archive_supplier):\n    needed=[]\n    for row in records:\n        path=safe_destination(data_root,row[\'relative_path\'])\n        if not path.is_file() or file_digest(path)!=row[\'sha256\']:\n            needed.append(row)\n    if not needed:\n        print(f\'All{len(records)} runtime TRAIN/development files already verified; no archive access.\')\n        return dict(required=len(records),restored=0,reused=len(records))\n    print(f\'Restoring/repairing {len(needed)} runtime copies; frozen Drive artifacts stay untouched.\',flush=True)\n    archive_path=archive_supplier()\n    pattern=re.compile(r\'^74034_3_En_4_MOESM1_ESM/(FVC2000|FVC2002|FVC2004)/Dbs/\'\n        r\'DB([123])_([AB])/([0-9]+_[0-9]+\\.(?:tif|bmp))$\',re.IGNORECASE)\n    with zipfile.ZipFile(archive_path) as archive:\n        mapping={}\n        for member in archive.infolist():\n            match=pattern.fullmatch(member.filename)\n            if not match:\n                continue\n            year,db,subset,filename=match.groups()\n            relative=f\'{year.upper()}/DB{db}_{subset.upper()}/{filename}\'\n            if ((member.external_attr>>16)&0o170000) not in (0,0o100000):\n                raise RuntimeError(f\'Unsafe non-regular/symlink archive member: {relative}\')\n            if relative in mapping:\n                raise RuntimeError(f\'Duplicate canonical archive member: {relative}\')\n            mapping[relative]=member\n        required_bytes=sum(mapping[row[\'relative_path\']].file_size for row in needed if row[\'relative_path\'] in mapping)\n        if shutil.disk_usage(data_root).free<required_bytes+64*1024*1024:\n            raise RuntimeError(\'Not enough ephemeral disk space to restore selected images.\')\n        for index,row in enumerate(needed,1):\n            relative=row[\'relative_path\']\n            if relative not in mapping:\n                raise RuntimeError(f\'Frozen image missing from official archive: {relative}\')\n            member=mapping[relative]\n            if member.file_size>32*1024*1024:\n                raise RuntimeError(f\'Unexpected fingerprint member size: {relative}\')\n            destination=safe_destination(data_root,relative)\n            destination.parent.mkdir(parents=True,exist_ok=True)\n            pending=destination.with_suffix(destination.suffix+\'.prepare.part\')\n            with archive.open(member) as src,pending.open(\'wb\') as dst:\n                shutil.copyfileobj(src,dst)\n            if file_digest(pending)!=row[\'sha256\']:\n                raise RuntimeError(f\'Frozen image checksum mismatch: {relative}; no replacement published.\')\n            pending.replace(destination)\n            if index%256==0 or index==len(needed):\n                print(f\'Verified restored images {index}/{len(needed)}\',flush=True)\n    return dict(required=len(records),restored=len(needed),reused=len(records)-len(needed))\n\n'
exec(compile(PREPARE_SOURCE,'<reset-prepare-helpers>','exec'))


In [ ]:

requirements,frozen_source33,split_protocol=frozen_environment(PROJECT_ROOT,'411576c19ecb440b8392')
package_report=restore_packages(requirements,PREPARE_DIR)
if package_report['restart_required']:
    raise RuntimeError('PACKAGES RESTORED. Use Runtime > Restart session, then Run All THIS00 notebook again. Already-loaded compiled modules cannot be safely hot-reloaded. Do not rerun training notebooks.')
import numpy as np
import pandas as pd
import scipy
import torch
from PIL import Image
import matplotlib
runtime_versions=dict(torch_version=str(torch.__version__),torch_cuda=str(torch.version.cuda),
    numpy_version=np.__version__,pillow_version=Image.__version__,scipy_version=scipy.__version__)
for key,value in runtime_versions.items():
    if frozen_source33[key]!=value:
        raise RuntimeError(f'Runtime import still differs from frozen{key}: {value}. Restart session and rerun00; no version fallback.')
print('Runtime matches frozen core libraries:',runtime_versions)
print('Python:',platform.python_version(),'| pandas:',pd.__version__,'| matplotlib:',matplotlib.__version__)


In [ ]:
RESUME_SOURCE = "def resume_download(url, partial, expected_bytes):\n    curl = shutil.which('curl')\n    if curl is None:\n        raise RuntimeError('curl is required for resumable downloads but is not installed.')\n    retryable_codes = {5, 6, 7, 18, 28, 35, 52, 55, 56, 92}\n    for attempt in range(1, 7):\n        size = partial.stat().st_size if partial.exists() else 0\n        if size == expected_bytes:\n            return\n        if size > expected_bytes:\n            raise RuntimeError(f'Partial file is oversized: {partial}. Inspect it before replacing it.')\n        print(f'Attempt {attempt}/6: resuming at {size / 2 ** 20:.1f} MiB', flush=True)\n        result = subprocess.run([curl, '--fail', '--location', '--show-error', '--progress-bar', '--continue-at', '-', '--connect-timeout', '30', '--speed-limit', '1024', '--speed-time', '60', '--max-time', '900', '--output', str(partial), url], check=False)\n        size = partial.stat().st_size if partial.exists() else 0\n        if result.returncode == 0 and size == expected_bytes:\n            return\n        if result.returncode == 0:\n            raise RuntimeError(f'Server returned an unexpected file size: {size}. Partial file retained.')\n        if result.returncode not in retryable_codes:\n            raise RuntimeError(f'curl failed with exit code {result.returncode}. Partial file retained at {partial}. See curl error above; HTTP or range errors require investigation.')\n        print(f'Connection interrupted (curl {result.returncode}); retained {size / 2 ** 20:.1f} MiB.')\n        if attempt < 6:\n            time.sleep(min(5 * attempt, 30))\n    raise RuntimeError(f'Download interrupted after 6 attempts. Rerun this cell to resume: {partial}')"
exec(compile(RESUME_SOURCE,'<existing02-resumable-download>','exec'))
ARCHIVE_URL=('https://static-content.springer.com/esm/'
    'chp%3A10.1007%2F978-3-030-83624-5_4/MediaObjects/74034_3_En_4_MOESM1_ESM.zip')
EXPECTED_ARCHIVE_BYTES=869928818
EXPECTED_ARCHIVE_MD5='0412227e3a8c99ab9b0022b8aef9912d'
records=locked_development_records(PROJECT_ROOT/'protocols'/'fvc_split_v1')
image_report=restore_development_images(records,DATA_ROOT,lambda:prepare_archive(
    PROJECT_ROOT,LOCAL_ROOT,ARCHIVE_URL,EXPECTED_ARCHIVE_BYTES,EXPECTED_ARCHIVE_MD5,resume_download))
print('Images ready:',image_report,'| no held-out/external images extracted or decoded.')


In [ ]:
PUBLIC_ASSET_SOURCE = "DMD_COMMIT = 'feb0ad40e5f340caebb457237d7ef21d5d5724e6'\nDMD_FILE_HASHES = {'models/model_zoo.py': '39f24735c03bcbbc0d51f6af75146bd3beb55a17bc25cf1a3b1c8f99bdd54b29', 'models/resnet.py': 'e6d28ed8a9730fc0ce5f1a98d464a129cd976ad91028e4516880908b42688e8d', 'models/units.py': '3c685be67e6ad57c5016cf44f75aa7660ba4675f17277faec470d44b3215a1aa', 'models/inception.py': 'bac60ed9f06c9e2e0869f73bb2bf987bbd3889856e4509f9a0cec5070d05702d', 'models/dataloader_densemnt.py': '70308d0b9a6258f5fb2eb815fa296a987190c0db8d9e5750c00c9df16e9468aa', 'evaluate_mnt.py': '84e7f4e62ebdaf718bc1266367cee5fa7680fd599d585c3cc9398f35a4e5b78f', 'README.md': '2b538fe64ef3510f60512a7b413eb04bbfc66a4c51b4fa00aa67e533eb84ee68', 'LICENCE': '4878c4e24ae4dd9878852724b4ae3313dc9a0599680d6552c1606bbc3d63830c', 'DMD.yaml': '12186bc0a33ab479d9c041aae8e35a180a286b98500c3d990475403732538b3d'}\nDMD_WEIGHT_SHA = '6d89dff9e5502cf4534b0fd9e014b3fb4a494a0be904015319c72edd9a7aff48'\nDMD_WEIGHT_BYTES = 179093287\nDMD_WEIGHT_URL = 'https://drive.usercontent.google.com/download?id=1iu8pbI2EjSkwOYsYczosJJDEhd1-JyzU&export=download&confirm=t'\nASSET_ROOT = PROJECT_ROOT / 'pretrained' / 'dmd_sd14_v1' / DMD_COMMIT\nASSET_ROOT.mkdir(parents=True, exist_ok=True)\n\ndef download_checked(url, path, sha, size=None):\n    if path.exists():\n        if file_digest(path) != sha or (size is not None and path.stat().st_size != size):\n            raise RuntimeError(f'Cached public asset failed integrity: {path}')\n        return\n    path.parent.mkdir(parents=True, exist_ok=True)\n    pending = path.with_suffix(path.suffix + '.part')\n    with urllib.request.urlopen(url, timeout=180) as response, pending.open('wb') as stream:\n        while True:\n            chunk = response.read(8 * 1024 * 1024)\n            if not chunk:\n                break\n            stream.write(chunk)\n    actual_sha = file_digest(pending)\n    actual_size = pending.stat().st_size\n    if actual_sha != sha or (size is not None and actual_size != size):\n        raise RuntimeError(f'Public download failed SHA256/size: expected_sha256={sha}, actual_sha256={actual_sha}, expected_bytes={size}, actual_bytes={actual_size}; retained partial: {pending}')\n    pending.replace(path)\nfor name, sha in DMD_FILE_HASHES.items():\n    download_checked(f'https://raw.githubusercontent.com/Yu-Yy/DMD/{DMD_COMMIT}/{name}', ASSET_ROOT / name, sha)\nWEIGHT_PATH = ASSET_ROOT / 'best_model.pth.tar'\ndownload_checked(DMD_WEIGHT_URL, WEIGHT_PATH, DMD_WEIGHT_SHA, DMD_WEIGHT_BYTES)"
exec(compile(PUBLIC_ASSET_SOURCE,'<existing33-public-asset-downloads>','exec'))
print('Verified pinned DMD assets. Existing files reused; no33 fitting/evaluation rerun.')


In [ ]:
NBIS_SOURCE = "NBIS_URL = 'https://nigos.nist.gov/nist/nbis/nbis_v5_0_0.zip'\nNBIS_SHA256 = '0adf8ab0f6b0e4208de50ca00ba21d3d77112ecd66288757ddfed21f6bee92c3'\nNBIS_BYTES = 52595795\nARCHIVE_DIR = PROJECT_ROOT / 'archives'\nARCHIVE_DIR.mkdir(parents=True, exist_ok=True)\nNBIS_ARCHIVE = ARCHIVE_DIR / 'nbis_v5_0_0.zip'\nmissing_tools = [name for name in ('curl', 'gcc', 'make', 'sh') if shutil.which(name) is None]\nif missing_tools:\n    raise RuntimeError(f'Missing build tools: {missing_tools}. Install required tools before retrying.')\nif not NBIS_ARCHIVE.exists():\n    partial = LOCAL_ROOT / 'nbis_v5_0_0.zip.part'\n    if not partial.exists() or partial.stat().st_size != NBIS_BYTES:\n        subprocess.run([\n            'curl', '--fail', '--location', '--continue-at', '-',\n            '--connect-timeout', '30', '--max-time', '600',\n            '--output', str(partial), NBIS_URL,\n        ], check=True)\n    if partial.stat().st_size != NBIS_BYTES or file_digest(partial) != NBIS_SHA256:\n        raise RuntimeError('NBIS download integrity failure. Partial file retained; do not build it.')\n    pending = NBIS_ARCHIVE.with_suffix('.zip.part')\n    shutil.copyfile(partial, pending)\n    if file_digest(pending) != NBIS_SHA256:\n        raise RuntimeError('NBIS Drive copy checksum mismatch.')\n    pending.replace(NBIS_ARCHIVE)\n    partial.unlink()\nif NBIS_ARCHIVE.stat().st_size != NBIS_BYTES or file_digest(NBIS_ARCHIVE) != NBIS_SHA256:\n    raise RuntimeError('Cached NBIS archive differs from the verified release.')\nprint('Official NBIS source verified:', NBIS_SHA256)"
NBIS_BUILD = "BUILD_ROOT = LOCAL_ROOT / 'tools' / 'nbis_5_0_0'\nSOURCE_ROOT = BUILD_ROOT / 'source'\nPREFIX = BUILD_ROOT / 'install'\nSOURCE = SOURCE_ROOT / 'Rel_5.0.0'\nMINDTCT = PREFIX / 'bin' / 'mindtct'\nBOZORTH3 = PREFIX / 'bin' / 'bozorth3'\nSTAMP = BUILD_ROOT / 'verified_build.json'\nbuild_commands = [\n    ['sh', 'setup.sh', str(PREFIX), '--without-X11', '--without-OPENJP2', '--64'],\n    ['make', 'config'],\n    ['make', 'it', 'CC=gcc -fcommon'],\n    ['make', 'install', 'LIBNBIS=no', 'CC=gcc -fcommon'],\n]\nbuild_signature = {'source_sha256': NBIS_SHA256, 'commands': build_commands}\nif STAMP.exists():\n    build_record = json.loads(STAMP.read_text(encoding='utf-8'))\n    if build_record['signature'] != build_signature:\n        raise RuntimeError('Cached NBIS build configuration differs; do not reuse it.')\n    for binary in (MINDTCT, BOZORTH3):\n        if not binary.is_file() or file_digest(binary) != build_record['binary_sha256'][binary.name]:\n            raise RuntimeError(f'Cached NBIS binary changed or is missing: {binary}')\n    print('Reusing verified local NBIS build.')\nelse:\n    SOURCE_ROOT.mkdir(parents=True, exist_ok=True)\n    PREFIX.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(NBIS_ARCHIVE) as archive:\n        for member in archive.infolist():\n            path = PurePosixPath(member.filename)\n            if path.is_absolute() or '..' in path.parts or '\\\\' in member.filename:\n                raise RuntimeError(f'Unsafe source archive path: {member.filename}')\n        archive.extractall(SOURCE_ROOT)\n    for path in SOURCE.rglob('*'):\n        if path.is_file() and (path.name == 'configure' or path.suffix == '.sh'):\n            path.chmod(path.stat().st_mode | 0o111)\n    build_log = WORK_DIR / 'build.log'\n    with build_log.open('w', encoding='utf-8') as output:\n        for command in build_commands:\n            print('Building:', ' '.join(command), flush=True)\n            result = subprocess.run(command, cwd=SOURCE, stdout=output, stderr=subprocess.STDOUT)\n            output.flush()\n            if result.returncode:\n                shutil.copyfile(build_log, REPORT_DIR / 'build.log')\n                print('\\n'.join(build_log.read_text(errors='replace').splitlines()[-60:]))\n                raise RuntimeError(f'NBIS build failed with code {result.returncode}. See saved build.log.')\n    shutil.copyfile(build_log, REPORT_DIR / 'build.log')\n    build_record = {\n        'signature': build_signature,\n        'binary_sha256': {binary.name: file_digest(binary) for binary in (MINDTCT, BOZORTH3)},\n        'compiler': subprocess.run(['gcc', '--version'], check=True, capture_output=True, text=True).stdout,\n        'platform': platform.platform(),\n    }\n    STAMP.write_text(json.dumps(build_record, indent=2), encoding='utf-8')\nversions = {}\nfor binary in (MINDTCT, BOZORTH3):\n    result = subprocess.run([str(binary), '-version'], check=True, capture_output=True, text=True)\n    versions[binary.name] = result.stdout + result.stderr\n    print(binary.name, versions[binary.name].strip())\n(REPORT_DIR / 'build_provenance.json').write_text(\n    json.dumps({**build_record, 'versions': versions}, indent=2), encoding='utf-8')"
if PREPARE_NBIS:
    WORK_DIR=PREPARE_DIR/'nbis_build'
    REPORT_DIR=PREPARE_DIR/'nbis_reports'
    WORK_DIR.mkdir(parents=True,exist_ok=True)
    REPORT_DIR.mkdir(parents=True,exist_ok=True)
    exec(compile(NBIS_SOURCE,'<existing04-source>','exec'))
    exec(compile(NBIS_BUILD,'<existing04-build-only>','exec'))
    nbis_status='verified_install_and_versions'
else:
    nbis_status='not_requested_not_needed_for35'
    print('NBIS skipped:35 does not use it. Set PREPARE_NBIS=True here before later NBIS matching.')


In [ ]:
TARGET_PREFLIGHT = {'environment': "import os\nos.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')\nimport hashlib\nimport importlib.util\nimport inspect\nimport json\nimport math\nimport shutil\nimport sys\nimport time\nimport types\nfrom pathlib import Path\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom PIL import Image\nfrom torch.utils.data import DataLoader, Subset\nfrom google.colab import drive\nDEVICE = torch.device('cpu')\ntorch.set_num_threads(min(4, os.cpu_count() or 1))\ntorch.backends.cudnn.benchmark = False\ntorch.backends.cudnn.deterministic = True\ntorch.backends.cudnn.allow_tf32 = False\ntorch.backends.cuda.matmul.allow_tf32 = False\ntorch.use_deterministic_algorithms(True)\ndrive.mount('/content/drive')\nPROJECT_ROOT = Path('/content/drive/MyDrive/FingerprintResearch')\nLOCAL_ROOT = Path('/content/fingerprint_research')\nDATA_ROOT = LOCAL_ROOT / 'data'\nSPLIT_DIR = PROJECT_ROOT / 'protocols' / 'fvc_split_v1'\nPAIR_DIR = PROJECT_ROOT / 'protocols' / 'nbis_validation_pairs_v1'\nSIM_DIR = PROJECT_ROOT / 'protocols' / 'structured_degradation_v1'\nM0_DIR = PROJECT_ROOT / 'experiments' / 'm0_unet_v1_seed20261110'\nM1_DIR = PROJECT_ROOT / 'experiments' / 'm1_shared_curriculum_v1_seed20261110' / 'dd1427cfad6621c577bf'\nSEED = 20261110\nRUN_SEEDS = {'dry_l1_seed20261110': 20261110}\nRUN_NAMES = tuple(RUN_SEEDS)\nM2_DIR = PROJECT_ROOT / 'experiments' / 'm2_structure_loss_v1_seed20261110' / '92579202157d85254b7f'\nSPECIALIZATION_ROOT = PROJECT_ROOT / 'experiments' / 'm5_dry_specialization_v1'\nLAMBDA_GRADIENT = 0.1\nIDENTITY_FRACTION = 0.1\n\ndef file_digest(path):\n    digest = hashlib.sha256()\n    with path.open('rb') as stream:\n        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):\n            digest.update(block)\n    return digest.hexdigest()\n\ndef json_bytes(value):\n    return (json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + '\\n').encode('utf-8')\n\ndef write_verified(path, data):\n    pending = path.with_suffix(path.suffix + '.part')\n    pending.write_bytes(data)\n    if pending.read_bytes() != data:\n        raise RuntimeError(f'Write verification failed: {path}')\n    pending.replace(path)\n\ndef write_csv(path, frame):\n    write_verified(path, frame.to_csv(index=False, lineterminator='\\n').encode('utf-8'))\n\ndef load_module(name, path):\n    spec = importlib.util.spec_from_file_location(name, path)\n    if spec is None or spec.loader is None:\n        raise RuntimeError(f'Cannot load module: {path}')\n    module = importlib.util.module_from_spec(spec)\n    sys.modules[name] = module\n    spec.loader.exec_module(module)\n    return module\n\ndef verify_files(directory, hashes):\n    for name, expected in hashes.items():\n        if file_digest(directory / name) != expected:\n            raise RuntimeError(f'Frozen artifact changed: {directory / name}')\nprint('Read-only prerequisites on CPU; DMD inference device selected below.')\nMATCHER_ROOT = PROJECT_ROOT / 'experiments' / 'risk_patch_matcher_v1'\nMATCHER_SETTINGS = {'epochs': 6, 'steps_per_epoch': 200, 'identities_per_batch': 12,\n    'learning_rate': 0.0002, 'embedding_dim': 128, 'held_groups_per_db': 8,\n    'quality_gate': {'clean_auc': 0.80, 'dry_auc': 0.70, 'clean_tar': 0.25, 'dry_tar': 0.10}}\n\nimport ast\nimport io\nimport sys\nimport subprocess\nimport urllib.request\nimport zipfile\nfrom scipy.optimize import linear_sum_assignment\nimport scipy\nimport torch.nn.functional as F\n\nINFERENCE_DEVICE = torch.device('cpu')\nSOURCE33_DIR = PROJECT_ROOT / 'diagnostics' / 'pretrained_dmd_feasibility_v1' / '411576c19ecb440b8392'\nprint('34: cached local scalar and one live input-gradient check on CPU; no training/GPU/NBIS.')\n\nSOURCE34_DIR = PROJECT_ROOT/'diagnostics'/'dmd_local_surrogate_v1'/'af30fd0800005636bbb8'\nCHECK_DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\nCHUNK_SIZE = 2\nprint('35 frozen matching; live check device:',CHECK_DEVICE,'| chunk size:',CHUNK_SIZE,'| NO TRAINING')\n", 'protocols': "split_protocol = json.loads((SPLIT_DIR / 'protocol.json').read_text(encoding='utf-8'))\nverify_files(SPLIT_DIR, split_protocol['file_sha256'])\nlocked = pd.read_csv(SPLIT_DIR / 'manifest_splits.csv', keep_default_na=False)\nm0_config = json.loads((M0_DIR / 'configuration.json').read_text(encoding='utf-8'))\nm1_config = json.loads((M1_DIR / 'configuration.json').read_text(encoding='utf-8'))\nm1_completion = json.loads((M1_DIR / 'completion.json').read_text(encoding='utf-8'))\nsim_config = json.loads((SIM_DIR / 'configuration.json').read_text(encoding='utf-8'))\npair_config = json.loads((PAIR_DIR / 'configuration.json').read_text(encoding='utf-8'))\nm0_sha, m1_sha = (file_digest(directory / 'configuration.json') for directory in (M0_DIR, M1_DIR))\nverify_files(M0_DIR, m0_config['source_file_sha256'])\nverify_files(M1_DIR, m1_config['source_file_sha256'])\nverify_files(M1_DIR, m1_completion['file_sha256'])\nif (m1_completion['status'] != 'shared_curriculum_control_complete' or m1_completion['configuration_sha256'] != m1_sha\n        or m1_config['m0_configuration_sha256'] != m0_sha):\n    raise RuntimeError('Frozen completed M1 provenance mismatch.')\nif any(config['split_protocol_sha256'] != file_digest(SPLIT_DIR / 'protocol.json') for config in (m0_config, sim_config, pair_config)):\n    raise RuntimeError('Split provenance differs across notebooks.')\nif (m0_config['simulator_configuration_sha256'] != file_digest(SIM_DIR / 'configuration.json')\n        or m1_config['simulator_configuration_sha256'] != file_digest(SIM_DIR / 'configuration.json')\n        or sim_config['normalization_configuration_sha256'] != file_digest(PAIR_DIR / 'configuration.json')\n        or file_digest(SIM_DIR / 'fingerprint_degradation.py') != sim_config['module_sha256']\n        or m0_config['simulator_module_sha256'] != sim_config['module_sha256']):\n    raise RuntimeError('Frozen simulator/normalization provenance mismatch.')\nfor config in (m0_config, m1_config):\n    if (str(torch.__version__) != config['torch_version'] or str(torch.version.cuda) != config['torch_cuda']\n            or np.__version__ != config['numpy_version'] or Image.__version__ != config['pillow_version']):\n        raise RuntimeError('Library versions differ from frozen training.')\nif Image.__version__ != pair_config['pillow_version'] or Image.__version__ != sim_config['pillow_version'] or np.__version__ != sim_config['numpy_version']:\n    raise RuntimeError('Normalization/simulator library versions differ.')\nfor name in ('restoration_baseline.py', 'train_sources.csv', 'validation_sources.csv'):\n    if file_digest(M1_DIR / name) != file_digest(M0_DIR / name):\n        raise RuntimeError(f'M1 model/source differs from M0: {name}')\nSETTINGS = dict(m1_config['settings'])\nEPOCHS = len(m1_config['epoch_conditions'])\nif (EPOCHS != sum(SETTINGS['phase_lengths']) or EPOCHS != m0_config['epochs']\n        or SETTINGS['base_channels'] != m0_config['base_channels'] or SETTINGS['patch_size'] != m0_config['patch_size']\n        or SETTINGS['batch_size'] != m0_config['batch_size'] or SETTINGS['learning_rate'] != m0_config['learning_rate']):\n    raise RuntimeError('Budget/architecture/optimizer differ from the frozen baseline.')\ntrain_sources = pd.read_csv(M0_DIR / 'train_sources.csv', keep_default_na=False)\nvalidation_sources = pd.read_csv(M0_DIR / 'validation_sources.csv', keep_default_na=False)\nfor split_name, frame in (('train', train_sources), ('validation', validation_sources)):\n    allowed = locked[locked['split'].eq(split_name)].set_index('relative_path')\n    if frame.empty or frame['relative_path'].duplicated().any() or frame.groupby(['dataset', 'db']).ngroups != 6 or set(frame['dataset']) != {'FVC2000', 'FVC2002'}:\n        raise RuntimeError(f'Invalid source-only {split_name} data.')\n    for record in frame.to_dict('records'):\n        relative = record['relative_path']\n        if (relative not in allowed.index or allowed.loc[relative, 'sha256'] != record['sha256']\n                or allowed.loc[relative, 'identity_key'] != record['identity_key']):\n            raise RuntimeError(f'Split/source mismatch: {relative}')\n        path = DATA_ROOT / relative\n        if split_name == 'train' and (not path.is_file() or file_digest(path) != record['sha256']):\n            raise RuntimeError(f'Image missing/changed: {path}. Restore with notebook 02.')\nif (set(train_sources['identity_key']) & set(validation_sources['identity_key'])\n        or set(train_sources['sha256']) & set(validation_sources['sha256'])\n        or not validation_sources.groupby(['dataset', 'db']).size().eq(12).all()):\n    raise RuntimeError('Train/validation overlap or changed validation coverage.')\ntraining = load_module('m2_original_model_v1', M0_DIR / 'restoration_baseline.py')\nsimulator = load_module('m2_original_simulator_v1', SIM_DIR / 'fingerprint_degradation.py')\ncurriculum_data = load_module('m2_frozen_task_data_v1', M1_DIR / 'curriculum_data.py')\nFROZEN_ENGINE_SOURCE = (M1_DIR / 'training_helpers.py.txt').read_bytes().decode('utf-8')\nSOURCE_COUNT = len(train_sources)\nall_conditions = [('identity', 'none')] + [(family, severity)\n    for family in ('dry_like', 'wet_like', 'generic') for severity in ('mild', 'moderate', 'severe')]\nprint('Verified unchanged sources:', SOURCE_COUNT, 'train /', len(validation_sources), 'validation.')\n\nallowed_train_paths = frozenset(train_sources.relative_path)\n", 'load': "\npointer = json.loads((MATCHER_ROOT / 'completed_experiment.json').read_text(encoding='utf-8'))\nMATCHER_DIR = Path(pointer['directory'])\nif (MATCHER_DIR.parent != MATCHER_ROOT\n        or file_digest(MATCHER_DIR / 'configuration.json') != pointer['configuration_sha256']\n        or file_digest(MATCHER_DIR / 'completion.json') != pointer['completion_sha256']):\n    raise RuntimeError('Notebook29 published pointer/integrity mismatch.')\nconfiguration = json.loads((MATCHER_DIR / 'configuration.json').read_text(encoding='utf-8'))\ncompletion = json.loads((MATCHER_DIR / 'completion.json').read_text(encoding='utf-8'))\nif (configuration['version'] != 'risk_patch_matcher_v1'\n        or configuration['settings'] != MATCHER_SETTINGS or configuration['seed'] != SEED\n        or completion['status'] != 'risk_matcher_fit_complete'\n        or completion['configuration_sha256'] != pointer['configuration_sha256']\n        or completion['completed_epochs'] != MATCHER_SETTINGS['epochs']\n        or configuration['selection'] != 'last fixed-budget checkpoint; no gate/validation epoch selection'\n        or configuration['train_sources_sha256'] != file_digest(M0_DIR / 'train_sources.csv')\n        or configuration['split_protocol_sha256'] != file_digest(SPLIT_DIR / 'protocol.json')\n        or configuration['simulator_sha256'] != file_digest(SIM_DIR / 'fingerprint_degradation.py')\n        or configuration['patch_size'] != SETTINGS['patch_size']):\n    raise RuntimeError('Expected unchanged completed29 fixed-budget matcher configuration.')\nverify_files(MATCHER_DIR, completion['file_sha256'])\nfor key, actual in (('torch_version', str(torch.__version__)), ('torch_cuda', str(torch.version.cuda)),\n        ('numpy_version', np.__version__), ('pillow_version', Image.__version__)):\n    if configuration[key] != actual:\n        raise RuntimeError(f'Matcher library version differs: {key}')\nmatcher_source = (MATCHER_DIR / 'matcher_source.py').read_bytes().decode('utf-8')\nif (hashlib.sha256(matcher_source.encode()).hexdigest() != configuration['model_source_sha256']\n        or file_digest(MATCHER_DIR / 'inner_partition.csv') != configuration['partition_sha256']\n        or file_digest(MATCHER_DIR / 'checkpoint_helpers.py.txt') != configuration['checkpoint_source_sha256']):\n    raise RuntimeError('Frozen matcher source/partition hash differs.')\nmatcher = types.ModuleType('read_only_failed29_matcher')\nmatcher.__dict__.update(globals())\nexec(compile(matcher_source, str(MATCHER_DIR / 'matcher_source.py'), 'exec'), matcher.__dict__)\npartition = pd.read_csv(MATCHER_DIR / 'inner_partition.csv', keep_default_na=False)\nexpected_partition = matcher.inner_partition(train_sources, MATCHER_SETTINGS['held_groups_per_db'])\npd.testing.assert_frame_equal(partition, expected_partition, check_dtype=False)\nlatest = json.loads((MATCHER_DIR / 'latest.json').read_text(encoding='utf-8'))\nif (latest != completion['checkpoint'] or Path(latest['file']).name != latest['file']\n        or latest['configuration_sha256'] != pointer['configuration_sha256']\n        or latest['completed_epochs'] != MATCHER_SETTINGS['epochs']\n        or file_digest(MATCHER_DIR / latest['file']) != latest['sha256']):\n    raise RuntimeError('LAST completed29 checkpoint differs.')\nstate = torch.load(MATCHER_DIR / latest['file'], map_location='cpu', weights_only=True)\nif (state['configuration_sha256'] != pointer['configuration_sha256']\n        or state['completed_epochs'] != MATCHER_SETTINGS['epochs']):\n    raise RuntimeError('Saved matcher payload differs.')\nwith torch.random.fork_rng(devices=[]):\n    model = matcher.PatchMatcher().to(DEVICE)\nmodel.load_state_dict(state['model'], strict=True)\nmodel.eval()\nfor parameter in model.parameters():\n    parameter.requires_grad_(False)\nif any(not torch.isfinite(value).all() for value in model.state_dict().values()):\n    raise RuntimeError('Nonfinite frozen matcher state.')\nsaved_state = {key: value.clone() for key, value in model.state_dict().items()}\nsaved_history = pd.DataFrame(state['history'])\nif saved_history.epoch.tolist() != list(range(1, MATCHER_SETTINGS['epochs'] + 1)):\n    raise RuntimeError('Matcher has not trained the full predefined budget.')\ndel state\ncalibration = pd.read_csv(MATCHER_DIR / 'calibration_scores.csv', float_precision='round_trip')\ngate = pd.read_csv(MATCHER_DIR / 'gate_scores.csv', float_precision='round_trip')\nsaved_gate = json.loads((MATCHER_DIR / 'quality_gate.json').read_text(encoding='utf-8'))\nsaved_thresholds = json.loads((MATCHER_DIR / 'thresholds.json').read_text(encoding='utf-8'))\nif saved_gate != completion['quality_gate']:\n    raise RuntimeError('Saved29 gate attestation differs.')\nprint('Read-only loaded matcher epoch:', latest['completed_epochs'], '| original gate passed:', saved_gate['passed'])\n", 'helpers': 'AUDIT_SOURCE = \'def panel_records(partition, split):\\n    if split not in (\\\'fit\\\', \\\'gate\\\', \\\'calibration\\\'):\\n        raise ValueError(\\\'Only explicit main-TRAIN inner partitions are supported.\\\')\\n    selected = partition[partition.matcher_split.eq(split)]\\n    if selected.empty:\\n        raise RuntimeError(\\\'Missing inner partition.\\\')\\n    if split == \\\'fit\\\':\\n        frames = []\\n        for (dataset, db), rows in selected.groupby([\\\'dataset\\\', \\\'db\\\'], sort=True):\\n            groups = sorted(rows.connected_group.unique(),\\n                key=lambda key: training.stable_seed(f\\\'{SEED}|audit_fit_panel|{key}\\\'))\\n            if len(groups) < MATCHER_SETTINGS[\\\'held_groups_per_db\\\']:\\n                raise RuntimeError(\\\'Insufficient fit groups for matched audit panel.\\\')\\n            frames.append(rows[rows.connected_group.isin(groups[:MATCHER_SETTINGS[\\\'held_groups_per_db\\\']])])\\n        selected = pd.concat(frames, ignore_index=True)\\n    galleries, probes = [], []\\n    for key, rows in selected.groupby(\\\'identity_key\\\', sort=True):\\n        records = rows.sort_values(\\\'relative_path\\\').drop_duplicates(\\\'sha256\\\').to_dict(\\\'records\\\')\\n        if len(records) < 3:\\n            raise RuntimeError(f\\\'Audit requires the same three byte-distinct source impressions: {key}\\\')\\n        galleries.append(records[0])\\n        probes.extend(records[1:3])\\n    return galleries, probes\\n\\ndef pair_metadata(partition, split):\\n    galleries, probes = panel_records(partition, split)\\n    rows = []\\n    for condition in (\\\'identity/none\\\', \\\'dry_like/mild\\\', \\\'dry_like/moderate\\\', \\\'dry_like/severe\\\'):\\n        for probe in probes:\\n            for gallery in galleries:\\n                if (probe[\\\'dataset\\\'], probe[\\\'db\\\']) != (gallery[\\\'dataset\\\'], gallery[\\\'db\\\']):\\n                    continue\\n                label = int(probe[\\\'identity_key\\\'] == gallery[\\\'identity_key\\\'])\\n                if not label and probe[\\\'connected_group\\\'] == gallery[\\\'connected_group\\\']:\\n                    continue\\n                rows.append(dict(dataset=probe[\\\'dataset\\\'], db=probe[\\\'db\\\'], condition=condition,\\n                    probe=probe[\\\'relative_path\\\'], gallery=gallery[\\\'relative_path\\\'], label=label))\\n    return pd.DataFrame(rows)\\n\\ndef verify_score_metadata(frame, partition, split):\\n    keys = [\\\'dataset\\\', \\\'db\\\', \\\'condition\\\', \\\'probe\\\', \\\'gallery\\\', \\\'label\\\']\\n    if (frame.empty or frame.duplicated(keys).any() or not np.isfinite(frame.score).all()\\n            or not frame.score.between(-1.00001, 1.00001).all() or set(frame.label) != {0, 1}):\\n        raise RuntimeError(\\\'Invalid saved inner verification scores.\\\')\\n    expected = pair_metadata(partition, split)\\n    pd.testing.assert_frame_equal(frame[keys].sort_values(keys).reset_index(drop=True),\\n        expected[keys].sort_values(keys).reset_index(drop=True), check_dtype=False)\\n\\ndef crop_details(record):\\n    relative = record[\\\'relative_path\\\']\\n    if relative not in allowed_train_paths:\\n        raise RuntimeError(\\\'Crop audit can read main TRAIN images only.\\\')\\n    dpi = pair_config[\\\'source_dpi\\\'][f"{record[\\\'dataset\\\']}_{record[\\\'db\\\']}"]\\n    reference = training.normalized_image(str(DATA_ROOT / relative), dpi)\\n    foreground = simulator.foreground_mask(reference)\\n    ys, xs = np.where(foreground)\\n    if not len(ys):\\n        raise RuntimeError(f\\\'No foreground for frozen crop: {relative}\\\')\\n    crop_seed = training.stable_seed(f\\\'{SEED}|0|{relative}|crop\\\')\\n    rng = np.random.default_rng(crop_seed)\\n    anchor = int(rng.integers(len(ys)))\\n    size = SETTINGS[\\\'patch_size\\\']\\n    height, width = reference.shape\\n    top = int(np.clip(int(ys[anchor]) - size // 2, 0, max(0, height - size)))\\n    left = int(np.clip(int(xs[anchor]) - size // 2, 0, max(0, width - size)))\\n    pad = ((0, max(0, size - height)), (0, max(0, size - width)))\\n    padded = np.pad(reference, pad, mode=\\\'reflect\\\') if any(value for pair in pad for value in pair) else reference\\n    patch = np.array(padded[top:top + size, left:left + size], dtype=np.float32, copy=True)\\n    actual = matcher.sample_patches([record], 0, [(\\\'identity\\\', \\\'none\\\')])[0, 0].numpy()\\n    if not np.array_equal(patch, actual):\\n        raise RuntimeError(\\\'Reconstructed crop is not byte-identical to frozen29 sampling.\\\')\\n    coverage = (min(size, height - top) * min(size, width - left)) / (height * width)\\n    metadata = dict(relative_path=relative, identity_key=record[\\\'identity_key\\\'], dataset=record[\\\'dataset\\\'],\\n        db=record[\\\'db\\\'], crop_seed=crop_seed, normalized_height=height, normalized_width=width,\\n        top=top, left=left, patch_size=size, padded_bottom=pad[0][1], padded_right=pad[1][1],\\n        crop_image_fraction=coverage, crop_foreground_fraction=float(\\n            foreground[top:min(top + size, height), left:min(left + size, width)].mean()),\\n        crop_sha256=hashlib.sha256(patch.astype(\\\'<f4\\\').tobytes()).hexdigest())\\n    return metadata, reference, patch\\n\\ndef describe_scores(values):\\n    values = np.asarray(values, dtype=np.float64)\\n    if not len(values) or not np.isfinite(values).all():\\n        raise RuntimeError(\\\'Nonempty finite scores required for descriptive summaries.\\\')\\n    return dict(count=len(values), mean=float(values.mean()), std=float(values.std(ddof=1)) if len(values) > 1 else None,\\n        q05=float(np.quantile(values, .05)), median=float(np.median(values)), q95=float(np.quantile(values, .95)))\\n\\ndef auc_against(genuine, impostors):\\n    genuine, impostors = np.asarray(genuine), np.asarray(impostors)\\n    if not len(genuine) or not len(impostors):\\n        raise RuntimeError(\\\'Both positive and negative scores required.\\\')\\n    return float(np.mean((genuine[:, None] > impostors).astype(float)\\n        + .5 * (genuine[:, None] == impostors)))\\n\\n@torch.inference_mode()\\ndef embed_panel(records, condition):\\n    features = []\\n    for start in range(0, len(records), 12):\\n        images = matcher.sample_patches(records[start:start + 12], 0, [condition]).to(DEVICE)\\n        value = model(images)\\n        if not torch.isfinite(value).all():\\n            raise RuntimeError(\\\'Nonfinite CPU embeddings.\\\')\\n        features.append(value.cpu())\\n    return torch.cat(features)\\n\\ndef summarize_panel(frame):\\n    summaries = []\\n    for (split, dataset, db, condition), group in frame.groupby(\\n            [\\\'inner_split\\\', \\\'dataset\\\', \\\'db\\\', \\\'condition\\\'], sort=True):\\n        threshold = saved_thresholds[f\\\'{dataset}_{db}\\\']\\n        positive = group[group.pool.eq(\\\'different_impression_genuine\\\')].score.to_numpy()\\n        for pool in (\\\'different_impression_genuine\\\', \\\'same_source_control\\\', \\\'same_db_impostor\\\', \\\'cross_db_impostor\\\'):\\n            values = group[group.pool.eq(pool)].score.to_numpy()\\n            summaries.append(dict(inner_split=split, dataset=dataset, db=db, condition=condition, pool=pool,\\n                threshold=threshold, acceptance_fraction=float(np.mean(values >= threshold)),\\n                auc_against_cross_impression_genuine=auc_against(positive, values)\\n                    if pool.endswith(\\\'impostor\\\') else None, **describe_scores(values)))\\n    return pd.DataFrame(summaries)\\n\'\nexec(compile(AUDIT_SOURCE, \'<matcher-failure-audit>\', \'exec\'))\n', 'replay': "\nverify_score_metadata(calibration, partition, 'calibration')\nverify_score_metadata(gate, partition, 'gate')\nthresholds, replay_metrics, replay_summary = matcher.gate_analysis(calibration, gate)\nif saved_thresholds != {f'{dataset}_{db}': value for (dataset, db), value in thresholds.items()}:\n    raise RuntimeError('Exact CSV replay changed original calibration thresholds.')\nif replay_summary != saved_gate:\n    raise RuntimeError('Saved gate summary differs from exact saved-score replay.')\nprior_metrics = pd.read_csv(MATCHER_DIR / 'gate_metrics.csv', float_precision='round_trip')\npd.testing.assert_frame_equal(replay_metrics, prior_metrics, check_dtype=False, check_exact=True)\nprint('Exact saved-score replay verified; gate UNCHANGED:')\nprint(json.dumps(replay_summary, indent=2))\nprint('Saved fixed-budget learning/runtime/memory history:')\nprint(saved_history.to_string(index=False))\n", 'source33': "\nEXPECTED_DMD_FILES = {'models/model_zoo.py': '39f24735c03bcbbc0d51f6af75146bd3beb55a17bc25cf1a3b1c8f99bdd54b29', 'models/resnet.py': 'e6d28ed8a9730fc0ce5f1a98d464a129cd976ad91028e4516880908b42688e8d', 'models/units.py': '3c685be67e6ad57c5016cf44f75aa7660ba4675f17277faec470d44b3215a1aa', 'models/inception.py': 'bac60ed9f06c9e2e0869f73bb2bf987bbd3889856e4509f9a0cec5070d05702d', 'models/dataloader_densemnt.py': '70308d0b9a6258f5fb2eb815fa296a987190c0db8d9e5750c00c9df16e9468aa', 'evaluate_mnt.py': '84e7f4e62ebdaf718bc1266367cee5fa7680fd599d585c3cc9398f35a4e5b78f', 'README.md': '2b538fe64ef3510f60512a7b413eb04bbfc66a4c51b4fa00aa67e533eb84ee68', 'LICENCE': '4878c4e24ae4dd9878852724b4ae3313dc9a0599680d6552c1606bbc3d63830c', 'DMD.yaml': '12186bc0a33ab479d9c041aae8e35a180a286b98500c3d990475403732538b3d'}\nEXPECTED_DMD_COMMIT = 'feb0ad40e5f340caebb457237d7ef21d5d5724e6'\nEXPECTED_WEIGHT_SHA = '6d89dff9e5502cf4534b0fd9e014b3fb4a494a0be904015319c72edd9a7aff48'\nEXPECTED_ADAPTER_SHA = '82cde5da161f7c37b58fe1ce069b460965c2a94eb175d12405840fffdc6c0919'\nsource_config = json.loads((SOURCE33_DIR / 'configuration.json').read_text(encoding='utf-8'))\nsource33_sha = file_digest(SOURCE33_DIR / 'configuration.json')\nsource_completion = json.loads((SOURCE33_DIR / 'completion.json').read_text(encoding='utf-8'))\nif (source_completion['status'] != 'pretrained_dmd_feasibility_complete'\n        or source_completion['configuration_sha256'] != source33_sha\n        or SOURCE33_DIR.name != source33_sha[:20]\n        or source_config['version'] != 'pretrained_dmd_feasibility_v1'\n        or source_config['source_commit'] != EXPECTED_DMD_COMMIT\n        or source_config['source_files'] != EXPECTED_DMD_FILES\n        or source_config['weight_sha256'] != EXPECTED_WEIGHT_SHA\n        or source_config['weight_bytes'] != 179093287\n        or source_config['max_anchors'] != 24 or source_config['minimum_anchors'] != 4\n        or source_config['adapter_sha256'] != EXPECTED_ADAPTER_SHA\n        or source_config['matcher29_configuration_sha256'] != pointer['configuration_sha256']\n        or source_config['matcher29_completion_sha256'] != pointer['completion_sha256']\n        or source_completion['permits_restoration_training'] is not False):\n    raise RuntimeError('Expected the unchanged completed33 source, not a different run/checkpoint.')\nverify_files(SOURCE33_DIR,source_completion['file_sha256'])\nfor name,value in [('torch_version',str(torch.__version__)),('torch_cuda',str(torch.version.cuda)),\n        ('numpy_version',np.__version__),('pillow_version',Image.__version__),('scipy_version',scipy.__version__)]:\n    if source_config[name] != value:\n        raise RuntimeError(f'Frozen33 library differs: {name}')\nASSET_ROOT = PROJECT_ROOT / 'pretrained' / 'dmd_sd14_v1' / EXPECTED_DMD_COMMIT\nverify_files(ASSET_ROOT,EXPECTED_DMD_FILES)\nWEIGHT_PATH = ASSET_ROOT / 'best_model.pth.tar'\nif file_digest(WEIGHT_PATH) != EXPECTED_WEIGHT_SHA or WEIGHT_PATH.stat().st_size != 179093287:\n    raise RuntimeError('Verified public DMD weights missing/changed; no unsafe fallback.')\nadapter33_source = (SOURCE33_DIR / 'adapter.py.txt').read_bytes().decode('utf-8')\nif hashlib.sha256(adapter33_source.encode()).hexdigest() != EXPECTED_ADAPTER_SHA:\n    raise RuntimeError('Frozen33 adapter changed.')\nexec(compile(adapter33_source,'<unchanged33-adapter>','exec'))\nlocal_score,_unused_lsar = load_scoring(ASSET_ROOT / 'evaluate_mnt.py')\ndel _unused_lsar\ndmd_model = load_dmd(ASSET_ROOT,WEIGHT_PATH)\ninitial_dmd_sha = state_sha(dmd_model)\nprint('Verified and loaded frozen33:',SOURCE33_DIR.name,'| previous LSA-R gate:',source_completion['quality']['passed'])\n", 'local-scalar': 'LOCAL_SOURCE = "def query_anchor_index(relative, count, epoch=0):\\n    if count < 4:\\n        raise RuntimeError(\'Local surrogate needs at least4 original anchors, same as33.\')\\n    return training.stable_seed(f\'{SEED}|{epoch}|{relative}|dmd_anchor_v1\') % count\\n\\ndef read_cached33(record, family, severity):\\n    relative = record[\'relative_path\']\\n    if relative not in allowed_train_paths:\\n        raise RuntimeError(\'Cached DMD local analysis is main TRAIN only.\')\\n    directory = SOURCE33_DIR / \'anchors\' / hashlib.sha256(relative.encode()).hexdigest()[:24]\\n    expected = dict(relative_path=relative,source_sha256=record[\'sha256\'],\\n        configuration_sha256=source33_sha,mindtct_sha256=source_config[\'nbis_build\'][\'binary_sha256\'][\'mindtct\'])\\n    if cached_bundle(directory,expected) is None:\\n        raise RuntimeError(f\'Missing completed33 anchor cache: {relative}. Run33; do not silently re-extract.\')\\n    with np.load(directory / \'anchors.npz\',allow_pickle=False) as archive:\\n        anchors = archive[\'anchors\'].copy()\\n        shape = archive[\'image_shape\'].copy()\\n    parsed = parse_nbis_xyt((directory / \'minutiae.xyt\').read_text(encoding=\'ascii\'),*shape)\\n    if not np.array_equal(anchors,parsed):\\n        raise RuntimeError(\'Frozen anchors disagree with their XYT source.\')\\n    if len(anchors) < 4:\\n        return dict(status=\'unmatchable_original_anchor_count\',anchors=anchors,feature=None,mask=None)\\n    feature_key = hashlib.sha256(f\'{relative}|{family}|{severity}\'.encode()).hexdigest()[:24]\\n    directory = SOURCE33_DIR / \'features\' / feature_key\\n    expected = dict(relative_path=relative,family=family,severity=severity,\\n        configuration_sha256=source33_sha,anchors_sha256=hashlib.sha256(anchors.tobytes()).hexdigest())\\n    if cached_bundle(directory,expected) is None:\\n        raise RuntimeError(f\'Missing completed33 feature cache: {relative}/{family}/{severity}. Run33.\')\\n    with np.load(directory / \'features.npz\',allow_pickle=False) as archive:\\n        feature = torch.from_numpy(archive[\'feature\'].copy())\\n        mask = torch.from_numpy(archive[\'mask\'].copy())\\n    if (feature.shape != (len(anchors),768) or mask.shape != (len(anchors),64)\\n            or not torch.isfinite(feature).all() or not torch.isfinite(mask).all()\\n            or not ((mask>=0)&(mask<=1)).all()):\\n        raise RuntimeError(\'Invalid cached33 local features.\')\\n    return dict(status=\'matched\',anchors=anchors,feature=feature,mask=mask)\\n\\ndef local_identity_score(query_feature,query_mask,gallery_feature,gallery_mask):\\n    if (query_feature.shape!=(1,768) or query_mask.shape!=(1,64)\\n            or gallery_feature.ndim!=2 or gallery_feature.shape[1]!=768\\n            or gallery_mask.shape!=(len(gallery_feature),64) or len(gallery_feature)<4\\n            or len(gallery_feature)>24):\\n        raise RuntimeError(\'Expected one local query and one4..24-anchor identity template.\')\\n    for value in (query_feature,query_mask,gallery_feature,gallery_mask):\\n        if not torch.isfinite(value).all():\\n            raise RuntimeError(\'Nonfinite local surrogate input.\')\\n    matrix = local_score(query_feature[None],gallery_feature[None],query_mask[None],gallery_mask[None],\\n        ndim_feat=12,Normalize=False,binary=False)[0]\\n    result = matrix.max(dim=1).values.squeeze(0)\\n    if not torch.isfinite(result):\\n        raise RuntimeError(\'Nonfinite local identity score.\')\\n    return result\\n\\ndef aligned_training_patch(image,anchor):\\n    if image.ndim!=4 or image.shape[:2]!=(1,1) or not torch.isfinite(image).all():\\n        raise RuntimeError(\'Expected one finite N1HW source.\')\\n    point = torch.as_tensor(anchor,device=image.device,dtype=image.dtype).flatten()\\n    if len(point)<3 or not torch.isfinite(point).all():\\n        raise RuntimeError(\'Expected a finite fixed original anchor.\')\\n    height,width = image.shape[-2:]\\n    yy,xx = torch.meshgrid(torch.arange(192,device=image.device,dtype=image.dtype)-96,\\n        torch.arange(192,device=image.device,dtype=image.dtype)-96,indexing=\'ij\')\\n    theta = torch.deg2rad(point[2])\\n    sx = point[0]+torch.cos(theta)*xx-torch.sin(theta)*yy\\n    sy = point[1]+torch.sin(theta)*xx+torch.cos(theta)*yy\\n    grid = torch.stack(((2*sx+1)/width-1,(2*sy+1)/height-1),dim=-1)[None]\\n    return F.grid_sample(image-.5,grid,mode=\'bilinear\',padding_mode=\'zeros\',align_corners=False)+.5\\n\\ndef local_prediction_features(prediction):\\n    if (prediction.ndim!=4 or prediction.shape[1:]!=(1,192,192)\\n            or not torch.isfinite(prediction).all()):\\n        raise RuntimeError(\'Expected finite N1x192x192 restoration outputs.\')\\n    output = dmd_model.get_embedding(2*prediction[:,:,32:160,32:160]-1)\\n    return output[\'feature\'],output[\'mask\']\\n\\ndef counterfactual_risks(before,after,threshold,temperature=.05,tail_fraction=.1):\\n    if (before.ndim!=1 or before.shape!=after.shape or not len(before)\\n            or not torch.isfinite(before).all() or not torch.isfinite(after).all()\\n            or not math.isfinite(threshold) or temperature<=0 or not 0<tail_fraction<=1):\\n        raise RuntimeError(\'Finite matched nonmated score vectors and valid risk constants required.\')\\n    increase = torch.relu(torch.sigmoid((after-threshold)/temperature)\\n        -torch.sigmoid((before.detach()-threshold)/temperature))\\n    count = max(1,math.ceil(tail_fraction*len(increase)))\\n    return increase.mean(),increase.topk(count).values.mean()\\n\\ndef selected_query(record,bundle):\\n    if bundle[\'status\']!=\'matched\':\\n        raise RuntimeError(\'Unmatchable queries must be explicitly counted upstream.\')\\n    index = query_anchor_index(record[\'relative_path\'],len(bundle[\'anchors\']),0)\\n    return index,bundle[\'feature\'][index:index+1],bundle[\'mask\'][index:index+1]\\n"\nexec(compile(LOCAL_SOURCE,\'<dmd-local-scalar>\',\'exec\'))\n', 'source34-and-warmstart': "\nsource34_config=json.loads((SOURCE34_DIR/'configuration.json').read_text(encoding='utf-8'))\nsource34_completion=json.loads((SOURCE34_DIR/'completion.json').read_text(encoding='utf-8'))\nsource34_sha=file_digest(SOURCE34_DIR/'configuration.json')\nif (source34_completion['status']!='dmd_local_surrogate_complete'\n        or SOURCE34_DIR.name!=source34_sha[:20]\n        or source34_completion['configuration_sha256']!=source34_sha\n        or source34_config['source33_configuration_sha256']!=source33_sha\n        or source34_completion['permits_legacy_30_31'] is not False):\n    raise RuntimeError('Expected unchanged completed34; no gate rewriting permitted.')\nverify_files(SOURCE34_DIR,source34_completion['file_sha256'])\nsource34_completion_sha=file_digest(SOURCE34_DIR/'completion.json')\nprint('34 remains unchanged. Quality:',source34_completion['quality']['passed'])\nSPECIALIZATION_ROOT=PROJECT_ROOT/'experiments'/'m5_dry_specialization_v1'\noriginal_pointer=json.loads((SPECIALIZATION_ROOT/'completed_experiment.json').read_text(encoding='utf-8'))\nORIGINAL_DIR=Path(original_pointer['directory'])\nif (ORIGINAL_DIR.parent!=SPECIALIZATION_ROOT\n        or file_digest(ORIGINAL_DIR/'configuration.json')!=original_pointer['configuration_sha256']\n        or file_digest(ORIGINAL_DIR/'completion.json')!=original_pointer['completion_sha256']):\n    raise RuntimeError('Original23 pointer/configuration/completion changed.')\noriginal_config=json.loads((ORIGINAL_DIR/'configuration.json').read_text(encoding='utf-8'))\noriginal_completion=json.loads((ORIGINAL_DIR/'completion.json').read_text(encoding='utf-8'))\nif (original_config['version']!='m5_dry_specialization_v1'\n        or original_completion['status']!='dry_specialization_complete'\n        or original_completion['configuration_sha256']!=original_pointer['configuration_sha256']\n        or original_config['seed']!=SEED\n        or original_config['initialization_seeds']!={'dry_l1_seed20261110':20261110}\n        or original_config['selection_metric']!='same fixed validation mean global L1; initial identity checkpoint included'\n        or original_config['losses']!={'dry_l1_seed20261110':'global L1'}\n        or original_config['gradient_lambda']!=0\n        or original_config['settings']!=SETTINGS):\n    raise RuntimeError('Expected the original23 common seed10 L1 warmstart, without reselection.')\nverify_files(ORIGINAL_DIR,original_completion['file_sha256'])\nverify_files(ORIGINAL_DIR,original_config['source_file_sha256'])\nfor name,path in [('restoration_baseline.py',M0_DIR/'restoration_baseline.py'),\n        ('train_sources.csv',M0_DIR/'train_sources.csv')]:\n    if original_config['source_file_sha256'][name]!=file_digest(path):\n        raise RuntimeError(f'Original23 source changed: {name}')\nif original_config['simulator_configuration_sha256']!=file_digest(SIM_DIR/'configuration.json'):\n    raise RuntimeError('Original23 simulator differs.')\nfor name,value in [('torch_version',str(torch.__version__)),('torch_cuda',str(torch.version.cuda)),\n        ('numpy_version',np.__version__),('pillow_version',Image.__version__)]:\n    if original_config[name]!=value:\n        raise RuntimeError(f'Original23 runtime differs: {name}')\noriginal_run=ORIGINAL_DIR/'dry_l1_seed20261110'\nif file_digest(original_run/'completion.json')!=original_completion['run_completion_sha256']['dry_l1_seed20261110']:\n    raise RuntimeError('Original23 selected run changed.')\noriginal_run_completion=json.loads((original_run/'completion.json').read_text(encoding='utf-8'))\nverify_files(original_run,original_run_completion['file_sha256'])\noriginal_best=json.loads((original_run/'best.json').read_text(encoding='utf-8'))\nif (original_best!=original_run_completion['result']['best_checkpoint']\n        or Path(original_best['file']).name!=original_best['file']\n        or original_best['configuration_sha256']!=original_pointer['configuration_sha256']\n        or file_digest(original_run/original_best['file'])!=original_best['sha256']\n        or original_run_completion['epochs_completed']!=8):\n    raise RuntimeError('Original23 best checkpoint/budget differs.')\npayload=torch.load(original_run/original_best['file'],map_location='cpu',weights_only=True)\nif (payload['configuration_sha256']!=original_pointer['configuration_sha256']\n        or payload['run']!='dry_l1_seed20261110' or payload['initialization_seed']!=20261110\n        or not 0<=payload['completed_epochs']<=8):\n    raise RuntimeError('Original23 safe payload differs.')\nwith torch.random.fork_rng(devices=[]):\n    restorer=training.RestorationUNet()\nrestorer.load_state_dict(payload['model'],strict=True)\ndel payload\nrestorer.eval()\nif sum(parameter.numel() for parameter in restorer.parameters())!=487009:\n    raise RuntimeError('Original23 restoration architecture differs.')\nif any(not torch.isfinite(value).all() for value in restorer.state_dict().values()):\n    raise RuntimeError('Nonfinite original23 weights.')\ninitial_restorer_sha=state_sha(restorer)\nprint('Verified original23 selected warmstart. No optimizer or updates.')\n"}
target_namespace={'__name__':'reset_prepare35_preflight'}
for stage,source in TARGET_PREFLIGHT.items():
    print('Checking35 prerequisite:',stage,flush=True)
    stage_started=time.perf_counter()
    exec(compile(source,f'<prepare35:{stage}>','exec'),target_namespace)
    print(f'Completed35 prerequisite: {stage} ({time.perf_counter()-stage_started:.1f}s)',flush=True)
print('Read-only35 prerequisites verified. Per-image33 caches are checked by35 on use; no duplicate Drive scan.',flush=True)


In [ ]:

gpu_name=None
if torch.cuda.is_available():
    gpu_name=torch.cuda.get_device_name(0)
    with torch.no_grad():
        values=torch.arange(16,dtype=torch.float32,device='cuda').reshape(4,4)
        result=values@values.T
        if not bool(torch.isfinite(result).all()):
            raise RuntimeError('CUDA arithmetic smoke check failed.')
        torch.cuda.synchronize()
    del values,result
    print('CUDA smoke check OK:',gpu_name)
else:
    print('CPU ready;35 can run, but its full-chain checks may be slower.')
report=dict(status='ready_for_35',python=platform.python_version(),runtime=runtime_versions,
    gpu=gpu_name,images=image_report,cache33_verification='deferred_to_35_on_use',nbis=nbis_status,
    target='35_frozen_lsar_full_chain_check.ipynb',
    preflight_sha256=hashlib.sha256(json.dumps(TARGET_PREFLIGHT,sort_keys=True).encode()).hexdigest(),
    source33_configuration_sha256=target_namespace['source33_sha'],
    source34_configuration_sha256=target_namespace['source34_sha'],
    original23_configuration_sha256=target_namespace['original_pointer']['configuration_sha256'],
    training_performed=False,legacy30_31_authorized=False)
READY_STAMP.write_text(json.dumps(report,indent=2,sort_keys=True)+'\n',encoding='utf-8')
del target_namespace
gc.collect()
print('PREPARE COMPLETE - READY FOR35')
print('Now open35 and Run All ON THIS SAME RUNTIME.')
print('Do not rerun01/02/04/23/29/33/34. No model was trained, no gate changed.')
print('Local readiness record:',READY_STAMP)
